[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/en_clase/S10_03_regularizacion_en_clase.ipynb)

# S10 · 03 — Entrenar bien: sobreajuste, regularización y callbacks · versión en clase

⏱ 55 min en clase · Sprint 10 · UD 4.3

**Contexto TurisData.** Una red que brilla en entrenamiento y falla con reservas nuevas es peor que inútil: da una falsa seguridad al cliente. Aquí aprenderás a **detectar** el sobreajuste con curvas de aprendizaje y a **controlarlo** con las herramientas habituales.

**Este cuaderno tiene 3 ejercicios** (1 a 3), los que se hacen en el aula. El efecto de la tasa de aprendizaje y del tamaño de lote, guardar y cargar un modelo, el mini-reto integrador y las preguntas de reflexión están en el cuaderno completo. Sus ejercicios 1, 2 y 3 son los 3 de este.

**Cómo usar este cuaderno** (tiene dos partes):

- **Parte A · Ejecutar** (aquí abajo): de arriba abajo, cada celda de código lleva encima su título con un símbolo:
    - ⚙️ **Preparación**: se ejecuta sola, pulsa ▶ y sigue.
    - 👀 **Ejemplo**: pulsa ▶, mira el resultado y responde las preguntas 🧠 (no hay que escribir código). Las respuestas están en la Parte B.
    - ✏️ **Ejercicio**: tu turno. Cambia los `...` y pulsa ▶. Si sale **✔**, responde las preguntas 🧠 y sigues. Para comprobar tu código, tienes la solución de referencia ✅ en la Parte B.
- **Parte B · Explicación** (al final): para cada celda, **qué hace**, **qué resultado debe salir** y, en los ejemplos, las **respuestas** a las preguntas 🧠 y, en los ejercicios, la **solución de referencia** ✅ para comprobar tu código. Ábrela cuando quieras entender una celda, comprobar tus respuestas o no te salga. Usa el índice de la izquierda (☰) para saltar.

# Parte A · Ejecutar

### ⚙️ A1 · PREPARACIÓN — se ejecuta sola

Pulsa ▶ y sigue. No hace falta leerla.

In [ ]:
#@title A1 · ▶ Ejecuta esta celda primero (no hace falta leerla)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import os, tempfile
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)
print("Listo ✔")

### 👀 A2 · EJEMPLO — Cargar y preparar los datos (solo 800 reservas para entrenar)

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A2).

1. ¿Cuántas filas tiene el entrenamiento pequeño y por qué se usa?
2. ¿Por qué el `ColumnTransformer` se ajusta solo con el entrenamiento?
3. ¿Para qué sirve la validación aquí?

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
num = ["antelacion_dias", "noches", "adultos", "ninos", "peticiones_especiales", "cliente_repetidor",
       "cancelaciones_previas", "mes_llegada", "precio_noche"]
cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[num + cat], df["cancelada"].to_numpy()
X_res, _, y_res, _ = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)   # se descarta la prueba
X_ent, X_val, y_ent, y_val = train_test_split(X_res, y_res, test_size=0.2, random_state=SEMILLA, stratify=y_res)
pre = ColumnTransformer([("n", StandardScaler(), num), ("c", OneHotEncoder(handle_unknown="ignore"), cat)])
A_ent = pre.fit_transform(X_ent).astype("float32")
A_val = pre.transform(X_val).astype("float32")

A_peq, y_peq = A_ent[:800], y_ent[:800]     # entrenamiento pequeño para provocar sobreajuste
print("Entrenamiento pequeño:", A_peq.shape, "· Entrenamiento completo:", A_ent.shape, "· Validación:", A_val.shape)

### 👀 A3 · EJEMPLO — Definir las funciones construir, entrenar y curvas

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A3).

1. ¿Cuántas neuronas ocultas tiene la red por defecto?
2. ¿Qué cambia al poner `dropout=0.4`?
3. ¿Por qué `entrenar` fija la semilla cada vez?

In [ ]:
def construir(n_entradas, capas=(256, 128), l2=0.0, dropout=0.0, batchnorm=False, tasa=1e-3):
    """MLP de clasificación con regularización opcional."""
    reg = keras.regularizers.l2(l2) if l2 > 0 else None
    m = keras.Sequential([keras.layers.Input(shape=(n_entradas,))])
    for u in capas:
        m.add(keras.layers.Dense(u, activation="relu", kernel_regularizer=reg, use_bias=not batchnorm))
        if batchnorm:
            m.add(keras.layers.BatchNormalization())
            m.add(keras.layers.Activation("relu"))   # (se vuelve a aplicar ReLU tras normalizar)
        if dropout > 0:
            m.add(keras.layers.Dropout(dropout))
    m.add(keras.layers.Dense(1, activation="sigmoid"))
    m.compile(optimizer=keras.optimizers.Adam(tasa), loss="binary_crossentropy",
              metrics=[keras.metrics.AUC(name="auc")])
    return m

def entrenar(A, y_, epocas=60, lote=32, callbacks=None, **cfg):
    keras.utils.set_random_seed(SEMILLA)
    m = construir(A.shape[1], **cfg)
    h = m.fit(A, y_, validation_data=(A_val, y_val), epochs=epocas, batch_size=lote,
              callbacks=callbacks, verbose=0)
    return m, h

def curvas(h, titulo, ax=None):
    ax = ax or plt.gca()
    ax.plot(h.history["loss"], label="entrenamiento")
    ax.plot(h.history["val_loss"], label="validación")
    ax.set_xlabel("época"); ax.set_ylabel("pérdida"); ax.set_title(titulo); ax.legend()

### 👀 A4 · EJEMPLO — Entrenar la red grande con pocos datos

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A4).

1. ¿En qué época tiene la validación su mejor pérdida y cuánto vale?
2. ¿Qué le pasa a la pérdida de entrenamiento y a la de validación después?
3. ¿Qué quiere decir «brecha de generalización»?

In [ ]:
m_base, h_base = entrenar(A_peq, y_peq, epocas=60)
plt.figure(figsize=(6, 4)); curvas(h_base, "Red grande y pocos datos: sobreajuste"); plt.show()

### ✏️ A5 · EJERCICIO 1 de 3 — Época óptima y brecha final

**Tu turno:** A partir de `h_base.history["val_loss"]` calcula: - `epoca_optima`: la época (contando desde 1) en la que la pérdida de validación fue **mínima**; - `brecha_final`: pérdida de validación de la última época **menos** pérdida de entrenamiento de la última época. Si `epoca_optima` es muy inferior al total de épocas, seguir entrenando solo empeoraba el modelo.

**Debe salir:** `Mejor época: 2 de 60 · brecha final = 2.11` y ✔ Ejercicio 1 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A5).

1. ¿Cuánto valen `epoca_optima` y `brecha_final`?
2. ¿Por qué se suma 1 al resultado de `argmin`?
3. Si `epoca_optima` es 2 de 60, ¿qué conclusión sacas?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A5, **después de intentarlo**.

In [ ]:
# pista: np.argmin(val_loss) + 1; y la resta de los últimos valores de val_loss y loss
perd_val = ...
perd_ent = ...
epoca_optima = ...
brecha_final = ...

assert 1 <= epoca_optima <= 60, "epoca_optima debe estar entre 1 y 60 (recuerda sumar 1 porque Python cuenta desde 0)"
assert brecha_final > 0.2, "Debería haber una brecha clara (validación bastante peor que entrenamiento). Revisa el signo de la resta"
print(f"Mejor época: {epoca_optima} de 60 · brecha final = {brecha_final:.2f}")
print("✔ Ejercicio 1 correcto")

### ✏️ A6 · EJERCICIO 2 de 3 — Entrenar con dropout

**Tu turno:** Entrena con `entrenar(A_peq, y_peq, ...)` una red con **dropout de 0.4** y guarda el modelo en `m_drop` y el historial en `h_drop`. Después comprueba en `n_dropout` cuántas capas `Dropout` tiene el modelo (usa `isinstance(capa, keras.layers.Dropout)`).

**Debe salir:** ✔ Ejercicio 2 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A6).

1. ¿Cuántas capas `Dropout` tiene el modelo y por qué?
2. ¿Qué hace una capa `Dropout(0.4)` durante el entrenamiento?
3. ¿Qué pérdida de validación final obtiene con dropout frente a sin regularizar?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A6, **después de intentarlo**.

In [ ]:
# pista: entrenar(A_peq, y_peq, epocas=60, dropout=0.4) y cuenta las capas Dropout de m_drop.layers
m_drop, h_drop = ...
n_dropout = ...

assert n_dropout == 2, "Con capas=(256, 128) debe haber 2 capas Dropout (una tras cada capa oculta)"
assert len(h_drop.history["loss"]) == 60, "Entrena 60 épocas"
print("✔ Ejercicio 2 correcto")

### 👀 A7 · EJEMPLO — Comparar L2, dropout y batch normalization

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A7).

1. ¿Qué configuración tiene la menor pérdida de validación final?
2. ¿Qué configuración tiene el mayor AUC de validación final?
3. ¿Por qué la «mejor pérdida val.» de L2 es peor que la de sin regularizar?

In [ ]:
configs = {
    "Sin regularizar": dict(),
    "L2 = 0.01": dict(l2=0.01),
    "Dropout 0.4": dict(dropout=0.4),
    "BatchNorm": dict(batchnorm=True),
    "L2 + Dropout": dict(l2=0.005, dropout=0.3),
}
resultados = {}
fig, ejes = plt.subplots(1, len(configs), figsize=(4 * len(configs), 3.6), sharey=True)
for eje, (nombre, cfg) in zip(ejes, configs.items()):
    m, h = entrenar(A_peq, y_peq, epocas=60, **cfg)
    resultados[nombre] = h.history
    curvas(h, nombre, eje)
plt.tight_layout(); plt.show()

resumen = pd.DataFrame({
    nombre: {"mejor pérdida val.": min(h["val_loss"]), "pérdida val. final": h["val_loss"][-1],
             "AUC val. final": h["val_auc"][-1], "AUC val. máximo": max(h["val_auc"])}
    for nombre, h in resultados.items()
}).T.round(3)
resumen

### ✏️ A8 · EJERCICIO 3 de 3 — Parada temprana y reducción de la tasa

**Tu turno:** Crea la lista `mis_callbacks` con un `EarlyStopping` (monitoriza `val_loss`, `patience=8`, `restore_best_weights=True`) y un `ReduceLROnPlateau` (monitoriza `val_loss`, `factor=0.5`, `patience=4`). Después entrena con `entrenar(A_peq, y_peq, epocas=100, callbacks=mis_callbacks, l2=0.005, dropout=0.3)` y guarda el resultado en `m_es` y `h_es`.

**Debe salir:** `Entrenó 26 de 100 épocas posibles. Mejor pérdida de validación: 0.743` y ✔ Ejercicio 3 correcto

Cambia los `...` y pulsa ▶.

🧠 **Autoejercicio:** cuando salga ✔, responde (respuestas en la Parte B, A8).

1. ¿Cuántas épocas entrena realmente y cuál fue la mejor?
2. ¿Por qué la parada ocurre en la época 26 y no en la 18?
3. ¿Qué pasaría sin `restore_best_weights=True`?

✅ ¿Dudas de tu código? Compáralo con la solución de referencia en la Parte B, A8, **después de intentarlo**.

In [ ]:
# pista: keras.callbacks.EarlyStopping(...) y keras.callbacks.ReduceLROnPlateau(...) dentro de una lista
mis_callbacks = [
    ...
]
m_es, h_es = ...

assert any(isinstance(c, keras.callbacks.EarlyStopping) for c in mis_callbacks), "Falta EarlyStopping"
assert any(isinstance(c, keras.callbacks.ReduceLROnPlateau) for c in mis_callbacks), "Falta ReduceLROnPlateau"
assert epocas_reales < 100, "La parada temprana debería haber cortado antes de las 100 épocas"
print(f"Entrenó {epocas_reales} de 100 épocas posibles. Mejor pérdida de validación: {min(h_es.history['val_loss']):.3f}")
print("✔ Ejercicio 3 correcto")

### 👀 A9 · EJEMPLO — Ver las curvas y cómo bajó la tasa de aprendizaje

Pulsa ▶ y mira lo que sale. **No hay que escribir nada.**

🧠 **Autoejercicio:** mira el resultado y responde (respuestas en la Parte B, A9).

1. ¿Cuántas veces bajó la tasa de aprendizaje y de qué valor a qué valor?
2. ¿Por qué el eje de la tasa está en escala logarítmica?
3. ¿Qué indica que la pérdida de validación deje de bajar?

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 3.6))
curvas(h_es, "Con EarlyStopping y ReduceLROnPlateau", ejes[0])
if "learning_rate" in h_es.history:
    ejes[1].plot(h_es.history["learning_rate"]); ejes[1].set_yscale("log")
    ejes[1].set_xlabel("época"); ejes[1].set_ylabel("tasa de aprendizaje"); ejes[1].set_title("Cómo bajó la tasa")
plt.tight_layout(); plt.show()

---

## 🏁 Fin del cuaderno de clase

El **[cuaderno completo](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S10/2_Practica/para_casa/S10_03_regularizacion.ipynb)** trae la tasa de aprendizaje y el tamaño de lote, guardar y cargar un modelo, el mini-reto integrador y las preguntas de reflexión, cada ejercicio con su solución plegada en la Parte B.

# Parte B · Explicación de cada celda y de cada resultado

Cada apartado lleva el **mismo número** que su celda de la Parte A (A1, A2…). Si una celda no te sale, mira su «Si falla».

#### A1 · Preparación — Preparar las herramientas

- **Qué hace:**
    - Carga `numpy`, `pandas`, `matplotlib`, `keras` y, de `scikit-learn`, la separación de datos y el preprocesado de columnas.
    - `os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"` silencia los mensajes técnicos de TensorFlow; `tempfile` servirá para guardar un modelo en una carpeta temporal.
    - `keras.utils.set_random_seed(SEMILLA)` fija las semillas para que los resultados se repitan.
- **Qué debe salir:**
    - `Listo ✔`. Importar TensorFlow tarda unos segundos.
- **Cómo leerlo:**
    - No hay que entender esta celda: solo ejecutarla **la primera**, y otra vez si reinicias el entorno.
- **Si falla:**
    - `NameError: name 'pd' is not defined` → no has ejecutado esta celda. Ejecútala y repite la que falló.
    - `ModuleNotFoundError: No module named 'tensorflow'` → fuera de Colab, instala TensorFlow en tu entorno.

## 0. Datos

#### A2 · Ejemplo — Cargar y preparar los datos (solo 800 reservas para entrenar)

**Contexto TurisData.** Una red que brilla en entrenamiento y falla con reservas nuevas es peor que inútil: da una falsa seguridad al cliente. Aquí aprenderás a **detectar** el sobreajuste con curvas de aprendizaje y a **controlarlo**.

Para ver bien el sobreajuste usaremos **solo 800 reservas de entrenamiento** (con pocos datos y una red grande,
memorizar es facilísimo). Validación: 960 reservas. La prueba final la reservamos para el reto.

- **Qué hace:**
    - Lee `reservas_turisdata.csv`, crea `mes_llegada` y separa las columnas numéricas (`num`) de las categóricas (`cat`). Se reserva el 20 % para la prueba (que **se descarta aquí**) y del resto el 20 % para validación.
    - El `ColumnTransformer` estandariza las numéricas y hace *one-hot* de las categóricas; se **ajusta solo con el entrenamiento** y se aplica a la validación.
    - `A_peq` son solo las **800 primeras** filas de entrenamiento: con pocos datos y una red grande, memorizar es facilísimo.
- **Qué debe salir:**
    - `Entrenamiento pequeño: (800, 35) · Entrenamiento completo: (3840, 35) · Validación: (960, 35)`.
- **Cómo leerlo:**
    - Cada reserva tiene 35 columnas. Usaremos 800 filas para provocar el sobreajuste y las 3 840 completas más adelante.
- **Si falla:**
    - `NameError: name 'dato' is not defined` → falta ejecutar la celda de preparación.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas filas tiene el entrenamiento pequeño y por qué se usa?* → **800**, para que una red grande tenga muchas más posibilidades de memorizarlas y se vea bien el sobreajuste.
2. *¿Por qué el `ColumnTransformer` se ajusta solo con el entrenamiento?* → Para **no usar información de la validación**: si no, habría fuga de datos y la evaluación sería demasiado optimista.
3. *¿Para qué sirve la validación aquí?* → Para **vigilar la generalización** durante el entrenamiento: es la que sube cuando hay sobreajuste.

</details>

## 1. Provocar y diagnosticar el sobreajuste

#### A3 · Ejemplo — Definir las funciones construir, entrenar y curvas

Tres funciones de ayuda que usaremos durante toda la práctica.

- **Qué hace:**
    - `construir(n_entradas, capas, l2, dropout, batchnorm, tasa)` crea un MLP de clasificación; por defecto tiene dos capas ocultas de **256 y 128** neuronas (42 241 parámetros con 35 entradas), y admite de forma opcional regularización **L2**, capas **Dropout** y **BatchNormalization**.
    - `entrenar(A, y_, epocas, lote, callbacks, **cfg)` fija la semilla, construye la red, la entrena con `fit` (la validación es siempre `A_val`) y devuelve el modelo y su historial.
    - `curvas(h, titulo, ax)` dibuja la pérdida de entrenamiento y la de validación por época.
- **Qué debe salir:**
    - No imprime nada: solo define las tres funciones.
- **Cómo leerlo:**
    - Cada técnica de regularización se activa con un argumento: `entrenar(A_peq, y_peq, dropout=0.4)`, `l2=0.01`, `batchnorm=True`…
- **Si falla:**
    - `NameError: name 'A_val' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas neuronas ocultas tiene la red por defecto?* → **256** en la primera capa y **128** en la segunda.
2. *¿Qué cambia al poner `dropout=0.4`?* → Se añade una capa `Dropout(0.4)` **tras cada capa oculta** (2 en total), que apaga el 40 % de las neuronas al azar en cada paso.
3. *¿Por qué `entrenar` fija la semilla cada vez?* → Para que, al comparar configuraciones, **la única diferencia sea la configuración** y no el azar de la inicialización.

</details>

#### A4 · Ejemplo — Entrenar la red grande con pocos datos

**Idea.** Una red con muchísimos parámetros para pocos datos acaba **aprendiéndose de memoria** los ejemplos,
incluido el ruido. Se reconoce porque:
- la pérdida de **entrenamiento** sigue bajando,
- la pérdida de **validación** baja al principio, toca un mínimo y **vuelve a subir**.

La distancia entre ambas curvas es la "brecha de generalización".

- **Qué hace:**
    - `entrenar(A_peq, y_peq, epocas=60)` entrena la red grande 60 épocas con las 800 reservas (`m_base`, `h_base`).
    - `curvas(...)` dibuja la pérdida de entrenamiento y la de validación por época.
- **Qué debe salir:**
    - Un gráfico: la pérdida de entrenamiento baja sin parar hasta ≈ 0,01; la de validación baja al principio (mínimo ≈ 0,61 en la época 2) y después **sube** hasta ≈ 2,1.
- **Cómo leerlo:**
    - **Lee las curvas.** Es el patrón típico de **sobreajuste**: la red se aprende de memoria las 800 reservas, incluido el ruido, y empeora con las demás. La distancia entre las dos curvas es la «brecha de generalización».
- **Si falla:**
    - `NameError: name 'entrenar' is not defined` → falta el ejemplo de las funciones. Tarda unos segundos en entrenar.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿En qué época tiene la validación su mejor pérdida y cuánto vale?* → En la **época 2**, con ≈ **0,61**.
2. *¿Qué le pasa a la pérdida de entrenamiento y a la de validación después?* → La de entrenamiento sigue **bajando** (≈ 0,01 al final) y la de validación **sube** (≈ 2,1).
3. *¿Qué quiere decir «brecha de generalización»?* → La **distancia** entre la pérdida de validación y la de entrenamiento: cuanto mayor, más sobreajuste.

</details>

#### A5 · Ejercicio 1 — Época óptima y brecha final

- **Qué hace:**
    - `perd_val` y `perd_ent` son las listas de pérdida por época de validación (`val_loss`) y entrenamiento (`loss`).
    - `np.argmin(perd_val)` da la **posición** del mínimo (empieza en 0), así que se suma 1 para contar épocas desde 1 (`epoca_optima`).
    - `brecha_final` resta la última pérdida de entrenamiento a la última de validación.
- **Qué debe salir:**
    - `Mejor época: 2 de 60 · brecha final = 2.11` y `✔ Ejercicio 1 correcto`.
- **Cómo leerlo:**
    - El mejor modelo estaba en la época 2: entrenar las otras 58 solo lo empeoró. La brecha de 2,11 es enorme.
- **Si falla:**
    - `epoca_optima debe estar entre 1 y 60 (recuerda sumar 1 ...)` → has olvidado el `+ 1`, o has usado el valor y no la posición.
    - `Debería haber una brecha clara ... Revisa el signo de la resta` → has restado al revés (entrenamiento − validación).
    - `NameError: name 'h_base' is not defined` → falta el ejemplo anterior.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuánto valen `epoca_optima` y `brecha_final`?* → **2** y **2,11**.
2. *¿Por qué se suma 1 al resultado de `argmin`?* → Porque Python cuenta desde **0**, pero las épocas se cuentan desde **1**.
3. *Si `epoca_optima` es 2 de 60, ¿qué conclusión sacas?* → Que **seguir entrenando solo empeoraba** el modelo: había que parar mucho antes (y regularizar).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
perd_val = h_base.history["val_loss"]
perd_ent = h_base.history["loss"]
epoca_optima = int(np.argmin(perd_val)) + 1
brecha_final = perd_val[-1] - perd_ent[-1]
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

## 2. Las herramientas de regularización

#### A6 · Ejercicio 2 — Entrenar con dropout

| Técnica | Idea en una frase | Cómo se pone en Keras |
|---|---|---|
| **L2** (*weight decay*) | Castiga los pesos grandes: la red prefiere soluciones "suaves" | `kernel_regularizer=keras.regularizers.l2(0.01)` |
| **Dropout** | En cada paso "apaga" al azar un % de neuronas: no puede depender de una sola | `keras.layers.Dropout(0.3)` |
| **Early stopping** | Parar cuando la validación deja de mejorar | callback `EarlyStopping` |
| **Batch normalization** | Reescala las activaciones en cada lote: entrena más estable y rápido; regulariza algo | `keras.layers.BatchNormalization()` |

La función `construir` de arriba ya admite `l2`, `dropout` y `batchnorm`. Vamos a probar cada una **por separado**.

- **Qué hace:**
    - `entrenar(A_peq, y_peq, epocas=60, dropout=0.4)` entrena la red con una capa `Dropout(0.4)` tras cada capa oculta y devuelve el modelo (`m_drop`) y su historial (`h_drop`).
    - `sum(isinstance(capa, keras.layers.Dropout) for capa in m_drop.layers)` cuenta cuántas capas del modelo son de tipo `Dropout`.
- **Qué debe salir:**
    - `✔ Ejercicio 2 correcto` (la celda no escribe nada más).
- **Cómo leerlo:**
    - Con dropout 0,4 la pérdida de validación toca su mínimo (≈ 0,61) hacia la época 4 y termina en ≈ 1,2, frente a ≈ 2,1 sin regularizar: **sobreajusta bastante menos**.
- **Si falla:**
    - `Con capas=(256, 128) debe haber 2 capas Dropout ...` → no has pasado `dropout=0.4` a `entrenar`.
    - `Entrena 60 épocas` → has cambiado `epocas`.
    - `NameError: name 'entrenar' is not defined` → falta el ejemplo de las funciones.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas capas `Dropout` tiene el modelo y por qué?* → **2**: una tras cada capa oculta (256 y 128).
2. *¿Qué hace una capa `Dropout(0.4)` durante el entrenamiento?* → En cada paso **apaga al azar el 40 % de las neuronas**, de modo que la red no puede depender de una sola.
3. *¿Qué pérdida de validación final obtiene con dropout frente a sin regularizar?* → ≈ **1,2** frente a ≈ **2,1**: sigue subiendo, pero mucho menos.

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
m_drop, h_drop = entrenar(A_peq, y_peq, epocas=60, dropout=0.4)
n_dropout = sum(isinstance(capa, keras.layers.Dropout) for capa in m_drop.layers)
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A7 · Ejemplo — Comparar L2, dropout y batch normalization

Probamos cada técnica **por separado** (y una combinación) con las mismas 800 reservas.

- **Qué hace:**
    - `configs` es un diccionario con cinco configuraciones: sin regularizar, L2 (0,01), dropout (0,4), `BatchNorm` y L2 + dropout.
    - Para cada una, `entrenar(...)` entrena 60 épocas, guarda su historial en `resultados` y dibuja sus curvas en una fila de cinco gráficos con el mismo eje y.
    - `resumen` es una tabla con la mejor pérdida de validación, la pérdida de validación final y el AUC final y máximo de cada configuración.
- **Qué debe salir:**
    - Cinco gráficos y una tabla (`pérdida val. final`): «Sin regularizar» ≈ 2,12 · «L2 = 0.01» ≈ 1,01 · «Dropout 0.4» ≈ 1,21 · «BatchNorm» ≈ 1,51 · «L2 + Dropout» ≈ 0,94. AUC de validación final: de ≈ 0,64 (sin regularizar) a ≈ 0,68 (L2 + dropout).
- **Cómo leerlo:**
    - Compara *pérdida val. final* con *mejor pérdida val.*: cuanto más se parecen, menos sobreajuste. Sin regularizar, la mejor fue 0,61 y la final 2,12; con L2 + dropout, 0,74 y 0,94.
    - La regularización rara vez «dispara» la métrica: lo que hace es **evitar que empeore**. Con tan pocos datos, muchas veces el mayor logro es no hacer el ridículo.
    - **Ojo con L2:** Keras suma la penalización de los pesos a **las dos** pérdidas (entrenamiento y validación), así que ambas salen más altas que sin regularizar, sobre todo al principio. Por eso la «mejor pérdida» de L2 (0,70) es peor que la de la red sin regularizar (0,61), aunque la final sea mejor.
- **Si falla:**
    - `NameError: name 'entrenar' is not defined` → faltan los ejemplos anteriores. Tarda unos segundos en entrenar las cinco redes.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Qué configuración tiene la menor pérdida de validación final?* → **L2 + Dropout**, con ≈ **0,94** (sin regularizar: ≈ 2,12).
2. *¿Qué configuración tiene el mayor AUC de validación final?* → **L2 + Dropout**, con ≈ **0,68**.
3. *¿Por qué la «mejor pérdida val.» de L2 es peor que la de sin regularizar?* → Porque Keras **suma la penalización L2 a la pérdida** (también a la de validación): sale más alta aunque la red generalice mejor.

</details>

## 3. Parada temprana y callbacks

#### A8 · Ejercicio 3 — Parada temprana y reducción de la tasa

**Idea.** Un *callback* es una función que Keras ejecuta al final de cada época (o lote). Los tres imprescindibles:
- `EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)`: para si la validación no mejora
  en 8 épocas seguidas y **devuelve los pesos de la mejor época**.
- `ReduceLROnPlateau(factor=0.5, patience=4)`: si se estanca, **baja la tasa de aprendizaje** a la mitad.
- `ModelCheckpoint(ruta, save_best_only=True)`: guarda el mejor modelo en disco.

(Existe también `TensorBoard`, un panel web para ver curvas; en Colab se activa con `%load_ext tensorboard`. Aquí basta con `matplotlib`.)

- **Qué hace:**
    - `EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)` para el entrenamiento si la pérdida de validación no mejora en 8 épocas seguidas y devuelve los pesos de la **mejor** época.
    - `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4)` baja la tasa de aprendizaje a la mitad si la validación se estanca 4 épocas.
    - `entrenar(..., epocas=100, callbacks=mis_callbacks, l2=0.005, dropout=0.3)` entrena como máximo 100 épocas con esos *callbacks* y con L2 y dropout (`m_es`, `h_es`).
- **Qué debe salir:**
    - `Entrenó 26 de 100 épocas posibles. Mejor pérdida de validación: 0.743` y `✔ Ejercicio 3 correcto`.
- **Cómo leerlo:**
    - La mejor época fue la 18; con `patience=8`, el entrenamiento paró a las 26 (18 + 8) y `restore_best_weights=True` devolvió los pesos de la época 18. Ahorra 74 épocas y deja el mejor modelo.
- **Si falla:**
    - `Falta EarlyStopping` o `Falta ReduceLROnPlateau` → falta alguno de los dos *callbacks* en `mis_callbacks`.
    - `La parada temprana debería haber cortado antes de las 100 épocas` → no has pasado `callbacks=mis_callbacks` a `entrenar`, o `patience` es demasiado alto.
    - `NameError: name 'm_es' is not defined` → falta el ejercicio (o el ejemplo de las funciones).

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas épocas entrena realmente y cuál fue la mejor?* → **26** de 100; la mejor fue la **18**.
2. *¿Por qué la parada ocurre en la época 26 y no en la 18?* → Porque `patience=8`: espera **8 épocas** sin mejora (18 + 8 = 26) antes de parar.
3. *¿Qué pasaría sin `restore_best_weights=True`?* → El modelo se quedaría con los pesos de la **última época** (la 26), ya algo peores, en lugar de los de la mejor (la 18).

</details>

<details><summary>✅ Comprueba tu código: solución de referencia (solo después de intentarlo)</summary>

```python
mis_callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
]
m_es, h_es = entrenar(A_peq, y_peq, epocas=100, callbacks=mis_callbacks, l2=0.005, dropout=0.3)

epocas_reales = len(h_es.history["loss"])
```

**Tu código puede ser distinto y valer**: lo que importa es que salga ✔ y que los valores coincidan con los de «Qué debe salir».

</details>

#### A9 · Ejemplo — Ver las curvas y cómo bajó la tasa de aprendizaje

- **Qué hace:**
    - `curvas(h_es, ...)` dibuja la pérdida de entrenamiento y de validación (gráfico izquierdo).
    - Si el historial incluye `learning_rate` (lo añade `ReduceLROnPlateau`), el gráfico derecho muestra su evolución en escala logarítmica.
- **Qué debe salir:**
    - A la izquierda, las curvas hasta la época 26. A la derecha, la tasa de aprendizaje: vale 0,001 al principio y baja a 0,0005 (una sola reducción).
- **Cómo leerlo:**
    - La pérdida de validación toca su mínimo (≈ 0,74, con L2 y dropout incluidos en la pérdida) en la época 18; el entrenamiento paró 8 épocas después.
- **Si falla:**
    - `NameError: name 'h_es' is not defined` → falta el ejercicio 3.

<details><summary>🧠 Respuestas al autoejercicio (después de intentarlo)</summary>

1. *¿Cuántas veces bajó la tasa de aprendizaje y de qué valor a qué valor?* → **Una vez**, de **0,001** a **0,0005**.
2. *¿Por qué el eje de la tasa está en escala logarítmica?* → Porque la tasa cambia **por factores** (mitad, mitad…), no por sumas: en escala log los saltos se ven iguales.
3. *¿Qué indica que la pérdida de validación deje de bajar?* → Que la red **ya no generaliza mejor**: es la señal para parar o para reducir la tasa.

</details>